# 179 — Cómo vigilar la frontera sin perseguir modas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — Clasificar por madurez

a) **standard**. Señales: especificación pública y versionada, adopción
multi-proveedor (tres implementaciones independientes), documentación oficial. Es el
único nivel donde adoptar es la acción por defecto si resuelve un problema existente.

b) **emerging**. Señales: tres semanas de vida, resultados autoevaluados, benchmark
propio (sin comparabilidad ni evaluador externo). Acción: anotar con fuente y fecha,
revisar en 4-8 semanas.

c) **current**. Señales: uso amplio y evaluaciones de terceros (más que emerging),
pero APIs inestables entre versiones menores (menos que standard). Acción: prototipo
acotado, aislando la dependencia detrás de una interfaz propia.

d) **deprecated**. Señal inequívoca: repositorio archivado con alternativa
recomendada. Acción: planificar migración y documentar la razón para que el equipo
no la reintroduzca por inercia.


In [ ]:
clasificacion = {
    "a": ("standard", "spec publica versionada + 3 implementaciones + docs"),
    "b": ("emerging", "3 semanas, autoevaluado, benchmark propio"),
    "c": ("current", "uso amplio y evals de terceros, pero APIs inestables"),
    "d": ("deprecated", "repositorio archivado con alternativa recomendada"),
}
niveles = {"emerging", "current", "standard", "deprecated"}
assert all(v[0] in niveles and v[1] for v in clasificacion.values())
print("clasificacion valida")


## Solución 2 — Rúbrica aplicada

a) - **Baseline: 2.** Sí hay comparación explícita y cuantificada (55 % → 78 %) sobre
     el mismo benchmark.
   - **Evaluador independiente: 0.** Evaluación del propio equipo; no hay tercero ni
     test oculto.
   - **Reproducibilidad: 0.** Código no publicado, sin receta verificable.
   - **Costo declarado: 0.** Ni tokens, ni tiempo, ni presupuesto de inferencia por
     issue — imposible saber si el salto viene del método o del gasto.

b) Total **2/8 → emerging**: anotar con fuente y fecha, no reestructurar nada.

c) Para subir de 0: (independiente) resultados sobre una variante verificada con
evaluación de terceros o un test oculto; (reproducible) pesos o API estable más una
receta que un externo haya ejecutado; (costo) tokens y tiempo medios por issue, con
el número de intentos por issue.

d) **Sí, empeora.** Si el benchmark es público y anterior al corte de entrenamiento,
la métrica puede estar contaminada por memorización: el 78 % deja de ser evidencia
de capacidad y pasa a ser una cota superior no interpretable. Pediría resultados
sobre issues posteriores al corte.


In [ ]:
rubrica = {"baseline": 2, "independiente": 0, "reproducible": 0, "costo": 0}
total = sum(rubrica.values())
nivel = "emerging" if total <= 3 else ("current" if total <= 6 else "standard")
print(total, "/8 ->", nivel)
assert total == 2 and nivel == "emerging"


## Solución 3 — El registro del repositorio

a) `frontier/current-topics.yaml` (versión `2026-07-29`) contiene 5 temas: dos
`standard` (MCP, A2A), dos `current` y uno `emerging`. El conteo exacto lo produce
la celda de código; lo relevante es que el registro completo cabe en una pantalla —
esa es la señal de que la vigilancia es barata por diseño.

b) **Auditables**: `source` (enlace a documentación primaria, permite a un tercero
verificar) y `reason` (por qué está en el registro, explicita el criterio).
**Caducables**: `reviewed` (fecha) y `maturity`; un script puede marcar como obsoleta
cualquier entrada con `reviewed` anterior a N semanas, y `version` fecha el registro
completo. Sin `reviewed` obligatorio, la obsolescencia solo se detectaría leyendo
todo a mano — es decir, no se detectaría.

c) Ejemplo de entrada bien formada (el `maturity` se justifica con señales):

```yaml
- id: ejemplo-tema
  name: Nombre del tema
  category: categoria-corta
  maturity: emerging
  reviewed: '2026-08-04'
  source: https://enlace-a-la-documentacion-primaria
  reason: Un paper y una implementación de referencia; sin evaluación de terceros.
```


In [ ]:
from pathlib import Path
from collections import Counter
import yaml

ruta = Path.cwd()
while not (ruta / "frontier" / "current-topics.yaml").exists() and ruta != ruta.parent:
    ruta = ruta.parent
datos = yaml.safe_load((ruta / "frontier" / "current-topics.yaml").read_text(encoding="utf-8"))
print("version del registro:", datos["version"])
print(Counter(t["maturity"] for t in datos["topics"]))
obligatorios = {"id", "name", "category", "maturity", "reviewed", "source", "reason"}
for t in datos["topics"]:
    assert obligatorios <= set(t), t["id"]
print("todas las entradas tienen fuente y fecha de revision")


## Solución 4 — Presupuesto de atención (referencia)

- **(4) Deprecación de una librería en producción — 2 h.** Máxima prioridad: es la
  única señal con costo asegurado si se ignora. Evaluar alternativa, estimar la
  migración y agendarla.
- **(1) Spec estable que resuelve un problema existente — 1.5 h.** Nivel `standard`
  y encaja con una necesidad real: leer la spec y decidir adopción. Es el único
  candidato a prototipo.
- **(2) Modelo con +3 % autoevaluado — 15 min.** `emerging`: anotar en el registro
  con fuente y fecha, revisar en 8 semanas. Sin evaluador independiente no merece
  más.
- **(3) Framework del que habla todo el mundo — 15 min.** La popularidad no es un
  criterio de la rúbrica; anotar y esperar a que aparezcan evaluaciones de terceros.

**Criterio de salida del prototipo (tema 1)**: "En 2 semanas y ≤10 h, la integración
vía el protocolo debe eliminar al menos dos adaptadores propios y pasar la suite de
integración existente; si no, se descarta y se documenta la razón en el registro."
Escrito ANTES de empezar: sin criterio previo, todo prototipo tiende a justificarse
a sí mismo.
